# Training prep 

In [3]:
# imports for all modules under src/preprocessing/*.py
from importlib import import_module
from pathlib import Path
import sys

# Make project src importable from this notebook location
project_root = Path.cwd().resolve().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Auto-import every preprocessing module and expose as globals
preprocessing_modules = {}
preprocessing_dir = src_path / "preprocessing"
for module_file in sorted(preprocessing_dir.glob("*.py")):
    if module_file.stem.startswith("_"):
        continue
    module = import_module(f"preprocessing.{module_file.stem}")
    preprocessing_modules[module_file.stem] = module
    globals()[module_file.stem] = module

print("Loaded preprocessing modules:", ", ".join(preprocessing_modules.keys()))

Loaded preprocessing modules: approvalDate, approvalFY, base_cleaning, city_bank, createJob, example, newExists, noemp, retainedJob


In [4]:
import pandas as pd
from pathlib import Path
import importlib

# Load df only if it is not already in memory
if "df" not in globals():
    project_root = Path.cwd().resolve().parent
    df = pd.read_csv(project_root / "data" / "train.csv")

In [ ]:
###  Limpieza base

In [ ]:
df = base_cleaning.clean_base_columns(df)

In [ ]:
###  NoEmp 
# seleccionar raw en caso de árboles.
# En caso de KNN o SVM seleccionar log
# Para análisis interpretativo seleccionar binning, probar en arboles también para ver si mejora o empeora el rendimiento.

In [ ]:
noemp_option = "raw"   # o "log" o "binning"
df = noemp.preprocess_noemp(df, option=noemp_option)

In [ ]:
### City y Bank

In [ ]:
preprocessor = city_bank.get_city_bank_encoder()

### NewExist

In [5]:
# calling src\preprocessing\newExists.py

# Reload module to pick up latest code changes
newExists = importlib.reload(newExists)

# Choose NewExist preprocessing path: 'A' or 'B'
newexist_option = "A" #===> is_new_business +  newexist_missing_or_invalid
# Option A: Create 'is_new_business' and 'newexist_missing_or_invalid' columns
# Option B: Create only 'is_new_business' column, and delete rows with missing/invalid values

# Apply preprocessing from the imported module
df_newexist = newExists.preprocess_newexist(
    df=df,
    option=newexist_option,
    source_col="NewExist",
)

print(f"NewExist option used: {newexist_option}")
print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_newexist):,}")

# Quick check of the generated columns
cols_to_show = ["NewExist", "is_new_business"]
if "newexist_missing_or_invalid" in df_newexist.columns:
    cols_to_show.append("newexist_missing_or_invalid")

display(df_newexist[cols_to_show].head(10))

# Show only rows flagged as missing/invalid in Option A
if "newexist_missing_or_invalid" in df_newexist.columns:
    display(
        df_newexist.loc[
            df_newexist["newexist_missing_or_invalid"] == 1,
            ["NewExist", "is_new_business", "newexist_missing_or_invalid"],
        ].head(20)
    )

NewExist option used: A
Rows before: 20,768
Rows after: 20,768


,NewExist,is_new_business,newexist_missing_or_invalid
0,1.0,0,0
1,2.0,1,0
2,2.0,1,0
3,2.0,1,0
4,1.0,0,0
5,2.0,1,0
6,1.0,0,0
7,1.0,0,0
8,1.0,0,0
9,1.0,0,0


,NewExist,is_new_business,newexist_missing_or_invalid
1080,0.0,0,1
2995,0.0,0,1
3226,0.0,0,1
3635,0.0,0,1
4503,0.0,0,1
6016,0.0,0,1
6630,0.0,0,1
10090,0.0,0,1
10473,<NA>,0,1
10730,<NA>,0,1


In [6]:
# calling src\preprocessing\newExists.py

# Reload module to pick up latest code changes
newExists = importlib.reload(newExists)

# Choose NewExist preprocessing path: 'A' or 'B'
newexist_option = "B"

# Apply preprocessing from the imported module
df_newexist = newExists.preprocess_newexist(
    df=df,
    option=newexist_option,
    source_col="NewExist",
)

print(f"NewExist option used: {newexist_option}")
print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_newexist):,}")

# Quick check of the generated columns
cols_to_show = ["NewExist", "is_new_business"]
if "newexist_missing_or_invalid" in df_newexist.columns:
    cols_to_show.append("newexist_missing_or_invalid")

display(df_newexist[cols_to_show].head(10))

# Show only rows flagged as missing/invalid in Option A
if "newexist_missing_or_invalid" in df_newexist.columns:
    display(
        df_newexist.loc[
            df_newexist["newexist_missing_or_invalid"] == 1,
            ["NewExist", "is_new_business", "newexist_missing_or_invalid"],
        ].head(20)
    )

NewExist option used: B
Rows before: 20,768
Rows after: 20,765


,NewExist,is_new_business
0,1.0,0
1,2.0,1
2,2.0,1
3,2.0,1
4,1.0,0
5,2.0,1
6,1.0,0
7,1.0,0
8,1.0,0
9,1.0,0


### CreateJob

In [7]:
# calling src\preprocessing\createJob.py

# Reload module to pick up latest code changes
createJob = importlib.reload(createJob)

# Choose CreateJob preprocessing path: 'A' (normalize) or 'B' (standardize)
createjob_option = "A"

# Apply preprocessing from the imported module
df_createjob = createJob.preprocess_createjob(
    df=df,
    option=createjob_option,
    source_col="CreateJob",
)

print(f"CreateJob option used: {createjob_option}")
print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_createjob):,}")

# Quick check of generated columns
cols_to_show = ["CreateJob"]
if "createjob_normalized" in df_createjob.columns:
    cols_to_show.append("createjob_normalized")
if "createjob_standardized" in df_createjob.columns:
    cols_to_show.append("createjob_standardized")

display(df_createjob[cols_to_show].head(10))

CreateJob option used: A
Rows before: 20,768
Rows after: 20,768


,CreateJob,createjob_normalized
0,0,0.0
1,1,0.000114
2,0,0.0
3,0,0.0
4,1,0.000114
5,0,0.0
6,1,0.000114
7,0,0.0
8,6,0.000682
9,1,0.000114


In [8]:
# calling src\preprocessing\createJob.py

# Reload module to pick up latest code changes
createJob = importlib.reload(createJob)

# Choose CreateJob preprocessing path: 'A' (normalize) or 'B' (standardize)
createjob_option = "B"

# Apply preprocessing from the imported module
df_createjob = createJob.preprocess_createjob(
    df=df,
    option=createjob_option,
    source_col="CreateJob",
)

print(f"CreateJob option used: {createjob_option}")
print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_createjob):,}")

# Quick check of generated columns
cols_to_show = ["CreateJob"]
if "createjob_normalized" in df_createjob.columns:
    cols_to_show.append("createjob_normalized")
if "createjob_standardized" in df_createjob.columns:
    cols_to_show.append("createjob_standardized")

display(df_createjob[cols_to_show].head(10))

CreateJob option used: B
Rows before: 20,768
Rows after: 20,768


,CreateJob,createjob_standardized
0,0,-0.03393
1,1,-0.028997
2,0,-0.03393
3,0,-0.03393
4,1,-0.028997
5,0,-0.03393
6,1,-0.028997
7,0,-0.03393
8,6,-0.004332
9,1,-0.028997


### RetainedJob

In [11]:
# calling src\preprocessing\retainedJob.py

# Reload module to pick up latest code changes
retainedJob = importlib.reload(retainedJob)

# Choose RetainedJob preprocessing path: 'A' (normalize) or 'B' (standardize)
retainedjob_option = "A"
# retainedjob_option = "B"

# Apply preprocessing from the imported module
df_retainedjob = retainedJob.preprocess_retainedjob(
    df=df,
    option=retainedjob_option,
    source_col="RetainedJob",
)

print(f"RetainedJob option used: {retainedjob_option}")
print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_retainedjob):,}")

# Quick check of generated columns
cols_to_show = ["RetainedJob"]
if "retainedjob_normalized" in df_retainedjob.columns:
    cols_to_show.append("retainedjob_normalized")
if "retainedjob_standardized" in df_retainedjob.columns:
    cols_to_show.append("retainedjob_standardized")

display(df_retainedjob[cols_to_show].head(10))

RetainedJob option used: A
Rows before: 20,768
Rows after: 20,768


,RetainedJob,retainedjob_normalized
0,0,0.0
1,1,0.000114
2,0,0.0
3,5,0.000568
4,3,0.000341
5,0,0.0
6,4,0.000455
7,0,0.0
8,0,0.0
9,6,0.000682


In [10]:
# calling src\preprocessing\createJob.py

# Reload module to pick up latest code changes
createJob = importlib.reload(createJob)

# Choose CreateJob preprocessing path: 'A' (normalize) or 'B' (standardize)
createjob_option = "B"

# Apply preprocessing from the imported module
df_createjob = createJob.preprocess_createjob(
    df=df,
    option=createjob_option,
    source_col="CreateJob",
)

print(f"CreateJob option used: {createjob_option}")
print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_createjob):,}")

# Quick check of generated columns
cols_to_show = ["CreateJob"]
if "createjob_normalized" in df_createjob.columns:
    cols_to_show.append("createjob_normalized")
if "createjob_standardized" in df_createjob.columns:
    cols_to_show.append("createjob_standardized")

display(df_createjob[cols_to_show].head(10))

CreateJob option used: B
Rows before: 20,768
Rows after: 20,768


,CreateJob,createjob_standardized
0,0,-0.03393
1,1,-0.028997
2,0,-0.03393
3,0,-0.03393
4,1,-0.028997
5,0,-0.03393
6,1,-0.028997
7,0,-0.03393
8,6,-0.004332
9,1,-0.028997


### ApprovalDate and ApprovalFY

In [12]:
import importlib
import sys
from pathlib import Path

# 1. Asegurar que el notebook encuentre la carpeta src (estilo de tu equipo)
project_root = Path.cwd().resolve().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# 2. Importar tus módulos
from preprocessing import approvalFY
from preprocessing import approvalDate


# =======================================================
# ApprovalFY
# =======================================================
approvalFY = importlib.reload(approvalFY)
approvalfy_option = "A"

df_approvalfy = approvalFY.preprocess_approvalfy(
    df=df,
    option=approvalfy_option,
    source_col="ApprovalFY",
)

print(f"Opción de ApprovalFY utilizada: {approvalfy_option}")
print(f"Filas antes: {len(df):,}")
print(f"Filas después: {len(df_approvalfy):,}")

if approvalfy_option == "A":
    print("\nOpción A seleccionada: La columna 'ApprovalFY' fue eliminada.")
    display(df_approvalfy.head(5))
elif approvalfy_option == "B":
    display(df_approvalfy[["ApprovalFY"]].head(10))


# =======================================================
# ApprovalDate
# =======================================================
approvalDate = importlib.reload(approvalDate)
approvaldate_option = "A"

df_approvaldate = approvalDate.preprocess_approvaldate(
    df=df, # ATENCIÓN: Usamos el df base para probar independientemente
    option=approvaldate_option,
    source_col="ApprovalDate",
)

print(f"\nOpción de ApprovalDate utilizada: {approvaldate_option}")
print(f"Filas antes: {len(df):,}")
print(f"Filas después: {len(df_approvaldate):,}")

if approvaldate_option == "A":
    cols_to_show = ["ApprovalYear", "ApprovalMonth"]
    print("\nOpción A seleccionada: Se crearon 'ApprovalYear' y 'ApprovalMonth'.")
    display(df_approvaldate[cols_to_show].head(10))
elif approvaldate_option == "B":
    cols_to_show = ["ApprovalDate"] 
    display(df_approvaldate[cols_to_show].head(10))

Opción de ApprovalFY utilizada: A
Filas antes: 20,768
Filas después: 20,768

Opción A seleccionada: La columna 'ApprovalFY' fue eliminada.


,id,LoanNr_ChkDgt,Name,City,State,Bank,BankState,ApprovalDate,NoEmp,NewExist,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept
0,64afe857c28,9448323000,MIDWEST CRANKSHAFT & ENGINE,HARVEY,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,9-Aug-96,28,1.0,0,0,1,0,N,N,31-Mar-97,"$600,000.00",$0.00,0
1,1705a7346c2,2854405007,"Iredesign, Limited",CHICAGO,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,10-Dec-07,1,2.0,1,1,0,1,N,N,31-Dec-07,"$25,400.00",$0.00,1
2,7439801ad8a,9300423010,PHILLY'S INC.,ROCHELLE,IL,BMO HARRIS BK NATL ASSOC,IL,23-May-96,6,2.0,0,0,1,0,N,Y,30-Sep-96,"$20,000.00",$0.00,1
3,a3f8f9d0611,4349265000,USA Laser Imaging Inc.,Loves park,IL,ALPINE BANK & TRUST CO.,IL,4-Nov-10,5,2.0,0,5,0,1,N,N,1-Mar-11,"$75,000.00",$0.00,1
4,71e4f243b5d,2433905006,"Dan Morrell, Inc.",LISLE,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,3-May-07,3,1.0,1,3,0,1,N,N,31-May-07,"$50,000.00",$0.00,0



Opción de ApprovalDate utilizada: A
Filas antes: 20,768
Filas después: 20,768

Opción A seleccionada: Se crearon 'ApprovalYear' y 'ApprovalMonth'.


,ApprovalYear,ApprovalMonth
0,1996,8
1,2007,12
2,1996,5
3,2010,11
4,2007,5
5,2003,3
6,2005,10
7,1993,2
8,1999,5
9,2004,3
